# Week 4 — Final Findings, Project Verification & Business Communication
**Project:** Food Delivery Data Analysis (`Food-Delivery-Data-Analysis-X-Kagnus`)  
**Dataset:** `data/processed/cleaned_food_delivery.csv` (`45,584` rows × `20` columns)  
**Objective:** Verify end-to-end project consistency across Weeks 1–3, synthesize 6 evidence-backed key findings, propose 3 practical operational recommendations for further investigation, and document limitations and next steps.


## 1. Project & Data Integrity Verification
**What & Why:** Before presenting final findings, we programmatically verify that all required repository files exist at their documented paths, confirm that raw and cleaned row counts match (`45,584` rows, 0 duplicates, 100% unique `ID`), validate all 5 exported PNG charts in `visualizations/`, and recompute every core summary table directly from `cleaned_food_delivery.csv`.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.image as mpimg

# 1. Verify all documented repository files exist and are non-empty
required_files = [
    "../data/raw/Food_delivery_Dataset.csv",
    "../data/processed/cleaned_food_delivery.csv",
    "../docs/cleaning_log.csv",
    "../notebooks/01_data_understanding_and_cleaning.ipynb",
    "../notebooks/02_exploratory_data_analysis.ipynb",
    "../notebooks/03_visualization.ipynb",
    "../visualizations/01_delivery_duration_distribution.png",
    "../visualizations/02_traffic_comparison.png",
    "../visualizations/03_weather_comparison.png",
    "../visualizations/04_distance_vs_duration.png",
    "../visualizations/05_vehicle_comparison.png",
    "../README.md",
    "../requirements.txt",
]
for path in required_files:
    assert os.path.exists(path) and os.path.getsize(path) > 0, f"Missing or empty: {path}"
print(f"Verified all {len(required_files)} required repository files exist.")

# 2. Load raw & cleaned datasets and verify structural integrity
raw_df = pd.read_csv("../data/raw/Food_delivery_Dataset.csv")
df = pd.read_csv("../data/processed/cleaned_food_delivery.csv")
cleaning_log = pd.read_csv("../docs/cleaning_log.csv")

assert len(raw_df) == len(df) == 45584, "Row count mismatch!"
assert df["ID"].nunique() == len(df) and df.duplicated().sum() == 0, "Primary key / duplicate check failed!"
print(f"Raw rows: {len(raw_df):,} | Cleaned rows: {len(df):,} | Unique IDs: {df['ID'].nunique():,} | Cleaning log entries: {len(cleaning_log)}")

# 3. Verify all 5 exported PNG charts are readable
viz_files = [p for p in required_files if "/visualizations/" in p]
viz_check = []
for p in viz_files:
    img = mpimg.imread(p)
    viz_check.append({
        "chart": os.path.basename(p),
        "size_kb": round(os.path.getsize(p) / 1024, 1),
        "dimensions_px": f"{img.shape[0]} x {img.shape[1]}"
    })
display(pd.DataFrame(viz_check))


Verified all 13 required repository files exist.


Raw rows: 45,584 | Cleaned rows: 45,584 | Unique IDs: 45,584 | Cleaning log entries: 17


,chart,size_kb,dimensions_px
0,01_delivery_duration_distribution.png,57.1,756 x 1400
1,02_traffic_comparison.png,56.4,756 x 1295
2,03_weather_comparison.png,77.7,756 x 1400
3,04_distance_vs_duration.png,352.4,806 x 1400
4,05_vehicle_comparison.png,63.7,785 x 1394


### Verification of Recomputed Metrics
**What & Why:** Recompute baseline duration statistics, straight-line Haversine distance correlations, and grouped summaries across traffic, weather, vehicle type, multi-deliveries, festival, and city tier so every numerical claim below is directly backed by live notebook output.


In [2]:
# Standardize column names to snake_case
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
TARGET = "time_taken_min"

# Compute Haversine straight-line distance (km) on valid coordinates
coord_cols = [
    "restaurant_latitude", "restaurant_longitude",
    "delivery_location_latitude", "delivery_location_longitude"
]
valid_coords = (df[coord_cols] > 0.1).all(axis=1) & df[coord_cols].notna().all(axis=1)
lat1, lon1, lat2, lon2 = [np.radians(df[c].abs()) for c in coord_cols]
a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
df["distance_km"] = np.where(valid_coords, 6371.0 * 2.0 * np.arcsin(np.sqrt(a)), np.nan)

# Baseline & distance correlation
t = df[TARGET]
d = df.dropna(subset=["distance_km", TARGET])
pearson_r = d["distance_km"].corr(d[TARGET])
spearman_rho = d["distance_km"].corr(d[TARGET], method="spearman")

print(f"Baseline -> N={t.count():,}, Mean={t.mean():.2f}, SD={t.std():.2f}, Median={t.median():.2f}, "
      f"Q1={t.quantile(0.25):.2f}, Q3={t.quantile(0.75):.2f}, IQR={t.quantile(0.75)-t.quantile(0.25):.2f}, Min-Max={t.min()}-{t.max()}")
print(f"15-35 min share: {((t >= 15) & (t <= 35)).mean() * 100:.1f}% | >=45 min count: {(t >= 45).sum():,} ({(t >= 45).mean() * 100:.2f}%)")
print(f"Distance -> Valid n={len(d):,}, Excluded={len(df)-len(d):,}, Range={d['distance_km'].min():.2f}-{d['distance_km'].max():.2f} km, "
      f"Mean={d['distance_km'].mean():.2f} km, Pearson r={pearson_r:.3f} (r^2={pearson_r**2:.3f}), Spearman rho={spearman_rho:.3f}")

def summarize_group(col: str, order=None) -> pd.DataFrame:
    """Return count, mean, median, and IQR of delivery duration for a grouping column."""
    tbl = df.dropna(subset=[col]).groupby(col)[TARGET].agg(
        count="count",
        mean="mean",
        median="median",
        q1=lambda s: s.quantile(0.25),
        q3=lambda s: s.quantile(0.75),
    )
    tbl["iqr"] = tbl["q3"] - tbl["q1"]
    if order:
        tbl = tbl.reindex(order)
    return tbl.round(2)

for label, col, order in [
    ("Traffic Density", "road_traffic_density", ["Low", "Medium", "High", "Jam"]),
    ("Weather Conditions", "weather_conditions", None),
    ("Vehicle Type", "type_of_vehicle", ["electric_scooter", "scooter", "bicycle", "motorcycle"]),
    ("Multiple Deliveries", "multiple_deliveries", None),
    ("Festival", "festival", None),
    ("City Tier", "city", None),
]:
    print(f"\n--- {label} ---")
    display(summarize_group(col, order))

print("\n--- Mean Distance (km) by Vehicle Type ---")
display(d.groupby("type_of_vehicle")["distance_km"].agg(["count", "mean", "median"]).round(2))


Baseline -> N=45,584, Mean=26.29, SD=9.38, Median=26.00, Q1=19.00, Q3=32.00, IQR=13.00, Min-Max=10-54
15-35 min share: 74.3% | >=45 min count: 1,803 (3.96%)
Distance -> Valid n=41,944, Excluded=3,640, Range=1.47-20.97 km, Mean=9.72 km, Pearson r=0.320 (r^2=0.103), Spearman rho=0.319

--- Traffic Density ---


,count,mean,median,q1,q3,iqr
road_traffic_density,,,,,,
Low,15476,21.27,20.0,16.0,26.0,10.0
Medium,10945,26.70,27.0,20.0,33.0,13.0
High,4423,27.24,27.0,22.0,32.0,10.0
Jam,14139,31.18,31.0,24.0,39.0,15.0



--- Weather Conditions ---


,count,mean,median,q1,q3,iqr
weather_conditions,,,,,,
Cloudy,7533,28.92,28.0,20.0,37.0,17.0
Fog,7653,28.91,28.0,20.0,37.0,17.0
Sandstorms,7494,25.88,26.0,20.0,31.0,11.0
Stormy,7584,25.87,26.0,20.0,31.0,11.0
Sunny,7282,21.86,20.0,16.0,26.0,10.0
Windy,7422,26.12,26.0,20.0,31.0,11.0



--- Vehicle Type ---


,count,mean,median,q1,q3,iqr
type_of_vehicle,,,,,,
electric_scooter,3814,24.47,24.0,17.0,30.00,13.00
scooter,15273,24.48,24.0,17.0,30.00,13.00
bicycle,68,26.43,26.0,19.0,32.25,13.25
motorcycle,26429,27.61,26.0,20.0,34.00,14.00


... (4 outputs left out)


--- City Tier ---


,count,mean,median,q1,q3,iqr
city,,,,,,
Metropolitan,34087,27.31,26.0,20.0,33.0,13.0
Semi-Urban,164,49.73,49.0,48.0,52.0,4.0
Urban,10133,22.98,22.0,16.0,28.0,12.0



--- Mean Distance (km) by Vehicle Type ---


,count,mean,median
type_of_vehicle,,,
bicycle,60,9.44,9.16
electric_scooter,3468,9.75,9.22
motorcycle,24390,9.69,9.19
scooter,14026,9.76,9.22


---

## 2. Key Findings (6 Evidence-Backed Observations)

### Finding 1: Delivery Duration Concentrates Around ~26 Minutes with a Mild Right Tail
- **Observation:** Across all 45,584 completed deliveries, turnaround times cluster tightly in the 15–35 minute range, accompanied by a smaller tail of prolonged orders.
- **Numerical Evidence:** Mean duration is **26.29 min** (SD = **9.38 min**), median is **26.00 min**, and IQR is **13.00 min** (Q1 = 19.00 min, Q3 = 32.00 min; full range 10–54 min). Exactly **74.3%** of orders finish within 15–35 minutes, while **3.96%** (**1,803** orders) take 45 minutes or more (`01_delivery_duration_distribution.png`).
- **Operational Interpretation:** Standard operations consistently deliver within a ~20–30 minute window. Service-level improvements should target the prolonged 45+ minute tail rather than redesigning normal-flow fulfillment.

---

### Finding 2: Road Traffic Congestion Shows the Largest Routine Delivery Delay (+11.0 Min Median Gap)
- **Observation:** Delivery duration increases monotonically from `Low` traffic to `Jam` conditions.
- **Numerical Evidence:** Median duration rises from **20.00 min** in `Low` traffic (mean 21.27 min, `n=15,476`) to **27.00 min** in `Medium` (mean 26.70 min, `n=10,945`) and `High` (mean 27.24 min, `n=4,423`), reaching **31.00 min** in `Jam` conditions (mean 31.18 min, `n=14,139`) — an **11.00-minute median** (+9.91-minute mean) gap (`02_traffic_comparison.png`). (`601` rows lack traffic labels.)
- **Operational Interpretation:** Heavy road congestion is strongly associated with slower deliveries. While congestion likely slows rider transit, this observational comparison does not isolate co-occurring peak-hour restaurant prep delays.

---

### Finding 3: Cloudy and Foggy Weather Coincide with ~7 Minutes Higher Average Delivery Times
- **Observation:** Average delivery times vary across weather conditions, with `Cloudy` and `Fog` showing the longest durations and `Sunny` the shortest.
- **Numerical Evidence:** Mean duration peaks in `Cloudy` (**28.92 min**, `n=7,533`) and `Fog` (**28.91 min**, `n=7,653`), sits near the overall average in `Windy` (**26.12 min**), `Sandstorms` (**25.88 min**), and `Stormy` (**25.87 min**), and drops to **21.86 min** in `Sunny` weather (`n=7,282`) — a **+7.06 min** (+32.3%) difference (`03_weather_comparison.png`). (`616` rows lack weather labels.)
- **Operational Interpretation:** Low-visibility or overcast conditions coincide with slower fulfillment, potentially reflecting cautious riding speeds or higher order volume; however, weather categories do not capture precipitation intensity or time-of-day overlap.

---

### Finding 4: Straight-Line Distance Explains Only ~10.3% of Delivery Duration Variance
- **Observation:** Longer trips tend to take longer, but the linear relationship is weak-to-moderate with substantial scatter at every distance.
- **Numerical Evidence:** Across **41,944** orders with valid GPS coordinates (1.47–20.97 km, mean **9.72 km**), Pearson **r = 0.320** (**r² = 0.103**) and Spearman **ρ = 0.319** (`04_distance_vs_duration.png`). Notably, **307** short trips (< 5 km) still took 45+ minutes, whereas **930** long trips (> 15 km) finished in 20 minutes or less. (`3,640` rows with placeholder coordinates were excluded.)
- **Operational Interpretation:** Straight-line distance leaves ~89.7% of duration variance unexplained, indicating that non-distance factors — such as traffic density, restaurant preparation time, and multi-order batching — dominate total delivery duration.

---

### Finding 5: Electric Scooters and Scooters Show Lower Central Durations Than Motorcycles
- **Observation:** Duration distributions differ by vehicle type even though average trip distances are nearly identical across fleets.
- **Numerical Evidence:** `electric_scooter` (`n=3,814`) and `scooter` (`n=15,273`) share a **24.00-minute** median (IQR = 13.00 min; means **24.47** and **24.48 min**), whereas `motorcycle` (`n=26,429`) records a **26.00-minute** median (IQR = 14.00 min; mean **27.61 min**) and `bicycle` (`n=68`) has a **26.00-minute** median (`05_vehicle_comparison.png`). Mean distances are balanced across motorcycles (**9.69 km**), scooters (**9.76 km**), and electric scooters (**9.44 km**).
- **Operational Interpretation:** Scooters and electric scooters are associated with ~2 min lower median (~3.1 min lower mean) delivery times than motorcycles without shorter trips. Because vehicle assignment is non-random, motorcycles may still be disproportionately routed during peak jams or multi-drop batches.

---

### Finding 6: Multi-Order Batching and Festival Days Show Steep Duration Escalations
- **Observation:** Bundling multiple orders on one run and operating during festivals coincide with the highest average durations in the dataset.
- **Numerical Evidence:** Mean duration rises from **22.88 min** (median 22.00 min, `n=14,094`) for `0` additional deliveries and **26.85 min** (`n=28,151`) for `1` delivery, to **40.45 min** (`n=1,985`) for `2` deliveries and **47.82 min** (median 48.00 min, `n=361`) for `3` deliveries (**+24.94 min** from 0 to 3). Similarly, `Festival = Yes` (`n=896`) averages **45.52 min** versus **25.98 min** on non-festival days (`n=44,460`).
- **Operational Interpretation:** High batch counts (2–3 concurrent deliveries) and festival surges are strongly associated with 40–48 minute turnaround times, pointing to a potential trade-off between batching efficiency and customer wait-time SLAs.

---

## 3. Operational Recommendations (Ideas to Investigate)

### Recommendation 1: Pilot Dynamic ETA Buffers & Pre-Dispatching During Jam Traffic and Fog/Cloudy Conditions
- **Observed Issue:** Delivery durations shift sharply upward during `Jam` traffic and `Cloudy`/`Fog` weather.
- **Supporting Evidence:** `Jam` orders (`n=14,139`) have a **31.00 min** median vs. **20.00 min** in `Low` traffic (+11.00 min); `Cloudy`/`Fog` orders (`n=15,186`) average **28.91–28.92 min** vs. **21.86 min** in `Sunny` weather (+7.06 min).
- **Proposed Action:** Investigate adding dynamic congestion/weather ETA buffers and testing earlier rider pre-dispatching during kitchen preparation when local conditions are flagged as `Jam` or `Fog`/`Cloudy`.
- **Operational Area That May Improve:** Customer ETA accuracy, on-time SLA adherence, and dispatch-to-pickup synchronization.
- **Additional Validation Needed:** Evaluate historical `Time_Order_picked - Time_Orderd` preparation intervals alongside traffic-weather cross-effects, and run a controlled A/B pilot in one `Metropolitan` zone before rolling out platform-wide.

---

### Recommendation 2: Evaluate Batching Caps (2–3 Orders) During Peak Traffic and Festival Surges
- **Observed Issue:** Multi-order batching at 2–3 drops and festival operations coincide with 40–48 minute delivery durations.
- **Supporting Evidence:** `2` deliveries average **40.45 min** (`n=1,985`) and `3` deliveries average **47.82 min** (`n=361`) compared to **22.88 min** at `0` (`n=14,094`); `Festival = Yes` (`n=896`) averages **45.52 min** vs. **25.98 min** on standard days.
- **Proposed Action:** Test a temporary cap on 3-order batching during `Jam` traffic or `Festival = Yes` periods in `Metropolitan` (`n=34,087`) and `Semi-Urban` (`n=164`, mean 49.73 min) zones to assess whether unbundling reduces 45+ minute tail deliveries.
- **Operational Area That May Improve:** Tail-latency SLA performance (reducing 45+ min orders) and customer retention during peak events.
- **Additional Validation Needed:** Check whether 2–3 batch orders already occur during peak kitchen backlogs, and run a small-scale pilot comparing courier utilization/cost per order against SLA gains.

---

### Recommendation 3: Audit GPS Telemetry Drops & Conduct Stratified Vehicle Performance Checks
- **Observed Issue:** Nearly 8% of trips have unusable placeholder GPS coordinates, and motorcycles average longer durations than scooters despite similar trip distances.
- **Supporting Evidence:** **3,640** rows (`7.99%`) lack valid GPS coordinates; `electric_scooter` (**24.47 min**) and `scooter` (**24.48 min**) average ~3.1 min faster than `motorcycle` (**27.61 min**) across nearly identical mean distances (`9.44–9.76 km`).
- **Proposed Action:** Audit rider-app GPS logging to reduce `(0, 0)`/placeholder coordinate failures, and run a stratified comparison of scooters vs. motorcycles within matched traffic, city, and batching tiers.
- **Operational Area That May Improve:** Spatial data reliability for routing/ETA models and urban last-mile fleet allocation.
- **Additional Validation Needed:** Verify through multivariate stratification whether scooter speed advantages hold after controlling for `Road_traffic_density`, `multiple_deliveries`, and `City`, rather than reflecting assignment bias.

---

## 4. Conclusion, Limitations & Next Steps

### Summary of the 4-Week Analysis
Across Weeks 1–4, we audited **45,584** raw delivery records (`01_data_understanding_and_cleaning.ipynb`), documented 18 data-quality actions in `docs/cleaning_log.csv` without fabricating or dropping valid records, quantified operational drivers in `02_exploratory_data_analysis.ipynb`, exported and interpreted 5 verified charts in `03_visualization.ipynb`, and synthesized 6 findings and 3 testable recommendations here.

### Key Limitations
1. **Association ≠ Causation:** All findings come from observational operational logs; observed differences across traffic, weather, vehicles, and batching do not prove direct causal impact without controlled experiments.
2. **Straight-Line Distance & Missing GPS:** Haversine distance underestimates true road-network travel distance, and **3,640** rows (`7.99%`) with sentinel GPS coordinates had to be excluded from distance analysis.
3. **Retained Missing Values:** Missing values in ratings (`1,908`), age (`1,854`), order time (`1,731`), city (`1,200`), multiple deliveries (`993`), weather (`616`), traffic (`601`), and festival (`228`) were preserved as `NaN` to avoid synthetic bias.
4. **Class Imbalance:** Smaller subgroups — `bicycle` (`n=68`), `Semi-Urban` (`n=164`), `3` multiple deliveries (`n=361`), and `Festival = Yes` (`n=896`) — have lower statistical precision than core segments.

### Possible Next Steps
- Decompose total delivery duration into **kitchen preparation / pickup wait** (`Time_Order_picked - Time_Orderd`) versus **on-road transit time**.
- Build a baseline regression or gradient-boosted model (with proper cross-validation and leakage prevention) to quantify the joint contribution of traffic, weather, batching, and distance.
- Fix GPS coordinate capture at order placement/pickup to recover spatial coverage on the missing 7.99% of trips.
